# AML Detection -- ExSTraQt-style pipeline

```
CSV
  |
  v
data_processing.py        (unchanged -- clean, encode, sort chronologically)
  |
  v
graph_construction.py     (transaction graph AND account graph)
  |
  v
feature_engineering.py    (existing behavioral/temporal/pair-history features)
community_detection.py    (Leiden + random-walk communities, on the account graph)
flow_features.py          (dispense/sink/passthrough + temporal flow)
  |
  v
feature_merge.py           (behavior + flow + community -> one dataframe)
  |
  v
model.py / train.py        (one XGBoost)
  |
  v
explain.py                 (SHAP)  +  aggregation.py (dashboard exports)
```
See `../README.md` for the full design writeup, including the `payment_format_idx` finding.

In [ ]:
import sys
sys.path.insert(0, "../src")

import pandas as pd
import plotly.express as px

import src.config
from src.data_processing import load_and_clean
from src.graph_construction import build_transaction_graph
from src.feature_engineering import engineer_all_features
from src.base_feature_columns import NUMERIC_FEATURE_COLUMNS, CATEGORICAL_FEATURE_COLUMNS
import src.train as T
from src.aggregation import build_transaction_view, aggregate_accounts

COMBINE_WITH_BASE_FEATURES = True
RESTRICT_TO_ACCOUNTS = src.config.RESTRICT_TO_ACCOUNTS  # see config.py -- None = every account

to copy cache files from the kaggle dataset into a working memory

In [ ]:
from pathlib import Path
import shutil

CACHE_DIR = Path("/kaggle/working/cache")
GRAPH_CACHE_DIR = CACHE_DIR / "graphs"
COMMUNITY_CACHE_DIR = CACHE_DIR / "communities"

GRAPH_CACHE_DIR.mkdir(parents=True, exist_ok=True)
COMMUNITY_CACHE_DIR.mkdir(parents=True, exist_ok=True)

DATASET = Path("/kaggle/input/datasets/aadityapandey1/exstraqt-ko-laagi")   # <-- replace after checking !ls

shutil.copy2(
    DATASET / "account_graph_edges_train.parquet",
    GRAPH_CACHE_DIR / "account_graph_edges_train.parquet",
)

shutil.copy2(
    DATASET / "account_graph_train.pkl",
    GRAPH_CACHE_DIR / "account_graph_train.pkl",
)

shutil.copy2(
    DATASET / "leiden_train.pkl",
    COMMUNITY_CACHE_DIR / "leiden_train.pkl",
)

In [ ]:
from pathlib import Path
import shutil

# ==========================
# DATA PATHS
# ==========================

TRAIN_CSV = Path("/kaggle/input/datasets/aadityapandey1/exstraqt-ko-laagi/HI-Small_Trans.csv")
TEST_CSV = Path("/kaggle/input/datasets/aadityapandey1/exstraqt-ko-laagi/LI-Small_Trans.csv")

# ==========================
# OUTPUT ROOT
# ==========================

OUTPUT_ROOT = Path("/kaggle/working")

CACHE_DIR = OUTPUT_ROOT / "cache"

GRAPH_CACHE_DIR = CACHE_DIR / "graphs"
COMMUNITY_CACHE_DIR = CACHE_DIR / "communities"
FEATURE_CACHE_DIR = CACHE_DIR / "features"
MODEL_CACHE_DIR = CACHE_DIR / "models"
EXPLAIN_CACHE_DIR = CACHE_DIR / "explainability"

EXPORT_DIR = OUTPUT_ROOT / "exports"

for p in [
    GRAPH_CACHE_DIR,
    COMMUNITY_CACHE_DIR,
    FEATURE_CACHE_DIR,
    MODEL_CACHE_DIR,
    EXPLAIN_CACHE_DIR,
    EXPORT_DIR,
]:
    p.mkdir(parents=True, exist_ok=True)

In [ ]:
DATASET = Path("/kaggle/input/datasets/aadityapandey1/exstraqt-ko-laagi")

for src, dst in [
    ("account_graph_edges_train.parquet", GRAPH_CACHE_DIR),
    ("account_graph_train.pkl", GRAPH_CACHE_DIR),
    ("leiden_train.pkl", COMMUNITY_CACHE_DIR),
]:
    source = DATASET / src
    target = dst / src

    if source.exists() and not target.exists():
        shutil.copy2(source, target)
        print(f"Copied {src}")

In [ ]:
df_train, vocabs = load_and_clean(str(TRAIN_CSV))
print(f"Train: {len(df_train):,} transactions, laundering rate {df_train['label'].mean():.3%}")

if COMBINE_WITH_BASE_FEATURES:
    _, src, dst, preds, succs = build_transaction_graph(df_train)
    df_train = engineer_all_features(df_train, preds, succs, src, dst)
    base_numeric, base_categorical = NUMERIC_FEATURE_COLUMNS, CATEGORICAL_FEATURE_COLUMNS
else:
    base_numeric, base_categorical = [], []

## Train
Community/flow/anomaly features are cached under `../cache/` at every stage (see `utils.py`) -- reruns after this point are fast unless you change the underlying transactions or `RESTRICT_TO_ACCOUNTS`.

In [ ]:
result = T.train(
    GRAPH_CACHE_DIR,
    COMMUNITY_CACHE_DIR,
    FEATURE_CACHE_DIR,
    df_train, val_frac=0.15,
    restrict_to_accounts=RESTRICT_TO_ACCOUNTS,
    base_numeric_columns=base_numeric, base_categorical_columns=base_categorical,
)
model, threshold = result["model"], result["threshold"]
print(result["metrics"])

In [ ]:
importance = model.feature_importance(top_n=25)
importance.to_csv(EXPORT_DIR / "feature_importance.csv", index=False)
px.bar(importance.sort_values("importance"), x="importance", y="feature", orientation="h",
       title="Feature importance (single ExSTraQt-style XGBoost)")

## SHAP: why was any ONE transaction flagged?
See `explain.py`. `model.feature_importance()` above is global; this is per-transaction.

In [ ]:
from src import explain
from src import feature_merge as fm

X_val = fm.prepare_feature_frame(result["df_val_joined"], result["feature_columns"]["numeric"], result["feature_columns"]["categorical"])
explainer = explain.build_explainer(model.booster_)
display(explain.summary(explainer, X_val, max_display=15))

flagged_idx = result["df_val_joined"].index[result["val_probs"] >= threshold]
if len(flagged_idx):
    display(explain.explain_transaction(explainer, X_val, flagged_idx[0]))

## Held-out file + dashboard export

In [ ]:
if TEST_CSV.exists():
    def gb(): return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss/1024/1024
    print(f"before load_and_clean(): {gb():.1f} GB")
    df_test, _ = load_and_clean(str(TEST_CSV), vocabs=vocabs)
    print(f"after load_and_clean(): {gb():.1f} GB")
    print(f"before engineer_all_features(): {gb():.1f} GB")
    if COMBINE_WITH_BASE_FEATURES:
        _, src_t, dst_t, preds_t, succs_t = build_transaction_graph(df_test)
        df_test = engineer_all_features(df_test, preds_t, succs_t, src_t, dst_t)
    print(f"after engineer_all_features(): {gb():.1f} GB")
    print(f"before score_holdout(): {gb():.1f} GB")
    holdout = T.score_holdout(GRAPH_CACHE_DIR, COMMUNITY_CACHE_DIR, FEATURE_CACHE_DIR,
                          model, df_train, df_test, result["feature_columns"], threshold,
                          restrict_to_accounts=RESTRICT_TO_ACCOUNTS,
                          build_features_from_holdout=True)
    print(holdout["metrics"])
    print(f"after score_holdout(): {gb():.1f} GB")

    transaction_view = build_transaction_view(holdout["df_holdout_view"], holdout["probs"], threshold=threshold)
    account_view = aggregate_accounts(holdout["df_holdout_joined"], holdout["probs"], threshold=threshold)
    transaction_view.to_csv(EXPORT_DIR / "transaction_view.csv", index=False)
    account_view = aggregate_accounts(holdout["df_holdout_view"], holdout["probs"], threshold=threshold)
    print("Wrote transaction_view.csv / account_view.csv / feature_importance.csv to data/exports/")
else:
    print(f"No held-out file at {TEST_CSV} -- skipping.")

In [ ]:
T.save_model(model, str(MODEL_CACHE_DIR / "exstraqt_model.json"))
print("Saved model to", MODEL_CACHE_DIR / "exstraqt_model.json")